# Benchmark results — exploratory analysis

Reads the committed `results/benchmark_results.csv` and produces a few
comparison plots. Re-run after every fresh benchmark sweep.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RESULTS = Path("..") / "results"
df = pd.read_csv(RESULTS / "benchmark_results.csv")
df.head()

In [ ]:
# Mean Qini by model with bootstrap CIs
summary = (
    df.groupby(["dataset", "model"])
      .agg(qini_mean=("qini", "mean"),
           qini_std=("qini", "std"),
           ci_low=("qini_ci_lower", "mean"),
           ci_hi=("qini_ci_upper", "mean"))
      .reset_index()
      .sort_values(["dataset", "qini_mean"], ascending=[True, False])
)
summary

In [ ]:
# Bar plot per dataset
for ds, sub in summary.groupby("dataset"):
    fig, ax = plt.subplots(figsize=(6, 0.5 * len(sub) + 1))
    s = sub.sort_values("qini_mean")
    err = [s["qini_mean"] - s["ci_low"], s["ci_hi"] - s["qini_mean"]]
    ax.barh(s["model"], s["qini_mean"], xerr=err, color="#1f77b4", alpha=0.85)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Qini (per-person)")
    ax.set_title(f"{ds}: model comparison")
    plt.tight_layout()
    plt.show()